# Chaotic Statistical Downscaling: run one station

This notebook runs the CSD pipeline of the `csd` package for one rain gauge:
read the climate-model cell (NetCDF), accumulate, calibrate (lag, amplitude, mutual false
neighbours), validate, synchronize, and write the downscaled series to NetCDF.

Edit the **Configuration** cell. If no station file is found, the notebook switches to a
**demo mode** with a synthetic station made from the model itself. Demo mode only shows that
the pieces fit together; its numbers say nothing about the method's skill.

Method: Duarte et al. (2018), <https://doi.org/10.29007/wkcx>.

## 1. Configuration

In [ ]:
from pathlib import Path

# --- climate model (MPI-ESM-MR, daily pr, NetCDF) ---------------------------------------
NC_FILES = [Path("../data/pr_day_MPI-ESM-MR_rcp45_r1i1p1_20060101-20091231.nc")]
# Several files are joined in order (historic first, scenario last; the later file wins
# where dates overlap), e.g. [historic_1850_2005.nc, rcp45_2006_2100.nc]

# --- station (daily table: date column + one column per station, dd/mm/yyyy) -------------
STATION_FILE = Path("../data/stations_daily.csv")      # your Bogota data
CATALOG_FILE = Path("../data/catalog.csv")             # name, latitude, longitude
STATION_NAME = None                                    # None -> first station in the table

# --- CSD settings -----------------------------------------------------------------------
USE_CHAOS_SELECTION = False        # True: choose step/dimension/delay with csd.chaos (needs snlda + nolds)
STEPS = [5, 7, 10, 15, 30]         # candidate accumulation steps (days) when selecting
STEP = 10                          # used when USE_CHAOS_SELECTION is False
DIMS = (3, 3)                      # (station, model) embedding dimension
DELAYS = (2, 2)                    # (station, model) delay, in accumulated steps
CAL_END = "2005-12-31"             # end of the calibration period
VAL_END = None                     # end of validation (None -> end of station record)
APP_END = None                     # end of the downscaled series (None -> end of model)

OUTPUT = Path("../output/downscaled_precipitation.nc")

## 2. Read the model cell and the station

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from csd import io, pipeline

DEMO = not (STATION_FILE.exists() and CATALOG_FILE.exists())

if DEMO:
    # Demo location: Bogota (4.6 N, 74.1 W)
    latitude, longitude, name = 4.6, -74.1, "demo"
else:
    table = io.read_station_table(STATION_FILE)
    catalog = io.read_catalog(CATALOG_FILE)
    name = STATION_NAME or table.columns[0]
    latitude, longitude = catalog.loc[name, ["latitude", "longitude"]]

model = None
for path in NC_FILES:
    part = io.read_model_cell(path, latitude, longitude)
    model = part if model is None else io.concat_model(model, part)
lat_b, lon_b = (tuple(round(float(v), 2) for v in part.attrs[k]) for k in ("lat_bounds", "lon_bounds"))
print(f"model cell {part.attrs['cell']}, lat {lat_b}, lon {lon_b}")
print(f"model: {model.index[0].date()} to {model.index[-1].date()}, mean {model.mean():.2f} mm/day")

if DEMO:
    print("DEMO MODE: no station file found; using a synthetic station (NOT real data).")
    rng = np.random.default_rng(0)
    station = (0.8 * model.shift(2).bfill() + rng.gamma(1.0, 1.0, len(model))).rename("station")
    CAL_END = str(model.index[len(model) // 2].date())
else:
    station = table[name].dropna().rename("station")
    print(f"station {name}: {station.index[0].date()} to {station.index[-1].date()}, mean {station.mean():.2f} mm/day")

## 3. Accumulation step, dimension and delay

With `USE_CHAOS_SELECTION = True` the first step at which station **and** model are
deterministic chaos is used (this part needs `snlda` and `nolds`). Otherwise the values
from the configuration cell are used.

In [ ]:
if USE_CHAOS_SELECTION:
    from csd import chaos
    selection = chaos.find_common_chaotic_step(station, model, STEPS)
    if selection is None:
        raise RuntimeError("No candidate step gives chaos for both series")
    step = selection.step
    dims = (selection.station.dimension, selection.model.dimension)
    delays = (selection.station.lag, selection.model.lag)
else:
    step, dims, delays = STEP, DIMS, DELAYS
print(f"step={step} days, dimensions={dims}, delays={delays}")

## 4. Calibration, validation and synchronization

In [ ]:
result = pipeline.downscale_station(
    station, model, step, dims, delays,
    cal_end=CAL_END, val_end=VAL_END, app_end=APP_END,
)
cal = result.calibration
print(f"calibration: lag = {cal.lag} steps, amplitude = {cal.amplitude:.3f}")
print(f"validation RMSE (accumulated {step}-day totals): {result.validation_rmse}")
print(f"downscaled blocks: {len(result.downscaled)}")

## 5. Plots

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
if len(result.validation):
    ax[0].plot(result.observed_validation, label="observed")
    ax[0].plot(result.validation, label="downscaled")
    ax[0].set_title("Validation period"); ax[0].legend()
ax[0].set_ylabel(f"mm per {step} days")
ax[1].plot(result.downscaled)
ax[1].set_title("Application period")
fig.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(4.5, 3.6))
ax.plot(result.calibration.lags, np.nanmin(result.calibration.sg, axis=0))
ax.set_xlabel("lag (steps)"); ax.set_ylabel("Sg (min over amplitudes)"); ax.set_title("Calibration")
plt.show()

## 6. Write the result to NetCDF

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
io.write_netcdf(OUTPUT, result.downscaled, name="pr", units=f"mm per {step} days")
check = io.read_netcdf_series(OUTPUT, "pr")
print(OUTPUT, len(check), "values; first:", check.index[0].date())